# Random Forest — PathSnap candidate classifier

Algorithm family: ensemble (bagging of decision trees).

This notebook:
1. Loads the shared candidate table (`data/processed/`). It uses the same features, labels and trip-level folds as every other model.
2. Runs 5-fold group cross-validation, grouped by trip, on the 80 training trips.
3. Refits on all training rows and evaluates on the 10 student-testing trips.
4. Saves `metrics.json`, `cv_folds.csv`, `model.joblib` and the plots into `models/random_forest/`.

Run the cells top to bottom. Runtime is about 5–10 min on a 4-core laptop with the settings below.

In [ ]:
import os, sys, json, time
from pathlib import Path
# find the repo root (the folder that contains config.yaml) and make `pathsnap` importable
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the pathsnap-ml-hackathon repo")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathsnap.dataset import FEATURES
from pathsnap import modeling as M
print("repo root:", ROOT)

## Settings
The rows a model is fitted on are subsampled inside each fold to keep runtime and model size manageable. Every trip is kept. Validation folds and the student-test set are always used in full.

In [ ]:
ROWS_PER_FOLD = 60_000          # 5 folds -> ~300k training rows (None = all ~1.48M rows)
PARAMS = dict(
    n_estimators=100,
    max_depth=20,                # limits overfitting and model file size
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=M.SEED,
)
OUT = ROOT / "models" / "random_forest"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
tr, te, tr_pts, te_pts = M.load_data()
print(f"labelled train rows: {len(tr):,}  (positive rate {tr.label.mean():.3f}, trips {tr.trip_id.nunique()})")
print(f"rows used for FITTING: up to {ROWS_PER_FOLD or 'all'} per fold; validation always uses full folds")
print(f"student-test rows: {len(te):,}  (trips {te.trip_id.nunique()})")
print(f"features ({len(FEATURES)}):", FEATURES)

## Model
Optimisation objective: each tree minimises Gini impurity. Rows are weighted by `trip_weight` so that every trip contributes equally. No feature scaling is needed for trees.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def make_model():
    return RandomForestClassifier(**PARAMS)

def fit_fn(model, df):
    return model.fit(df[FEATURES], df["label"], sample_weight=df["trip_weight"].values)

## 5-fold group cross-validation (trip-level)

In [ ]:
cv = M.cross_validate(make_model, fit_fn, tr, tr_pts, ROWS_PER_FOLD)
cv[["fold", "top1_accuracy", "train_top1", "f1_macro", "precision_macro", "recall_macro", "log_loss", "fit_seconds"]].round(4)

In [ ]:
print("CV Top-1     : %.4f ± %.4f" % (cv.top1_accuracy.mean(), cv.top1_accuracy.std()))
print("CV macro F1  : %.4f ± %.4f" % (cv.f1_macro.mean(), cv.f1_macro.std()))
print("CV log loss  : %.4f" % cv.log_loss.mean())
print("train Top-1  : %.4f   (gap to CV = overfitting evidence)" % cv.train_top1.mean())
M.plot_cv_folds(cv, "Random Forest: train vs validation Top-1 per fold", OUT / "cv_folds.png")

## Final fit on all training rows → student-testing trips

In [ ]:
model, scores, test, train_s, infer_ms = M.fit_and_test(make_model, fit_fn, tr, te, te_pts, ROWS_PER_FOLD)
print(f"train time: {train_s:.1f}s   inference: {infer_ms:.3f} ms per 1000 candidate rows")
pd.Series({k: v for k, v in test.items() if k != "confusion_matrix"}).to_frame("student_test")

In [ ]:
M.plot_confusion(test["confusion_matrix"], "Random Forest — student-test (threshold 0.5)", OUT / "confusion_matrix.png")

## Feature importance (impurity-based)

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(6, 8))
imp.plot.barh(ax=ax, color="#2e7d32")
ax.set(title="Random Forest feature importance", xlabel="mean decrease in impurity")
plt.tight_layout(); plt.savefig(OUT / "feature_importance.png", dpi=130); plt.show()
imp.sort_values(ascending=False).to_csv(OUT / "feature_importance.csv")

## Save results

In [ ]:
res = M.save_run(OUT, "random_forest", "Ensemble (bagging)", PARAMS, cv, test, train_s, infer_ms,
                 model=model, rows_per_fold=ROWS_PER_FOLD)
print(json.dumps({k: v for k, v in res.items() if k != "student_test"}, indent=1, default=str))

## Notes for the report (fill in from YOUR results above)
- Why this algorithm suits PathSnap, and its strengths and limitations as observed here.
- Hyperparameters and initial values: see `PARAMS`. Objective: Gini impurity per tree.
- Student-test Top-1, macro F1 and log loss: see the table above.
- Over- or underfitting: compare `train_top1` with the validation Top-1, and look at the variation across folds.